# Feature Analysis

This notebook will analyze the final feature set selected in the previous notebook and determine any remaining preprocessing considerations before model training.

Planned area:

- **Feature distributions** — analyze feature ranges, scale differences, skewness, and extreme values to determine appropriate preprocessing.

The goal is to make **actionable preprocessing decisions for the training pipeline**, without repeating the feature-selection analysis.

In [1]:
import numpy as np
import pandas as pd

from src.common.constants import FINAL_SELECTED_FEATURES
from src.modeling.data_utils import load_modeling_data, split_modeling_data

In [2]:
df = load_modeling_data()

train_df, val_df, test_df = split_modeling_data(df)

print(f"Train:, {len(train_df)}, {train_df["ts"].min()}, →, {train_df["ts"].max()}")

Train:, 1001, 2022-08-11 00:00:00+00:00, →, 2025-05-31 00:00:00+00:00


## Experiment 1 — Feature Distribution Analysis

**Goal:** Understand the distributions and scales of the final 23 features to identify potential preprocessing needs for downstream models.

**Potential checks:**
- Feature ranges and scale differences
- Distribution shape and skewness
- Extreme or suspicious values

**Focus:** Determine whether scaling or transformations may be worth considering for Ridge Regression and the Dense NN. No feature selection will be performed here.

In [3]:
X_train = train_df[FINAL_SELECTED_FEATURES].copy()

distribution_analysis = pd.DataFrame({
    "min": X_train.min(),
    "max": X_train.max(),
    "mean": X_train.mean(),
    "std": X_train.std(),
    "median": X_train.median(),
    "skewness": X_train.skew(),
    "unique": X_train.nunique(),
})

distribution_analysis["range"] = (
    distribution_analysis["max"] - distribution_analysis["min"]
)

distribution_analysis["cv"] = (
    distribution_analysis["std"] / distribution_analysis["mean"].abs()
)

distribution_analysis.round(3)

,min,max,mean,std,median,skewness,unique,range,cv
pm25_today,0.662,264.850,71.175,43.142,57.000,1.511,981,264.187,0.606
pm10_today,0.942,639.979,108.674,61.313,93.175,1.788,990,639.037,0.564
o3_today,26.792,161.083,86.624,22.974,84.583,0.362,803,134.292,0.265
co_today,59.208,7611.917,1306.547,1063.398,969.292,2.544,997,7552.708,0.814
no2_today,0.000,184.100,44.674,26.682,39.225,1.718,971,184.100,0.597
so2_today,0.325,64.396,18.660,10.295,15.400,1.473,943,64.071,0.552
aqi_roll_mean_7,91.708,284.262,154.340,38.446,149.089,0.896,980,192.554,0.249
pm25_lag_1,0.662,264.850,71.152,43.325,56.946,1.507,982,264.187,0.609
pm25_lag_2,0.662,264.850,71.177,43.418,57.000,1.507,982,264.187,0.610
pm25_lag_3,0.662,264.850,71.238,43.601,57.000,1.508,981,264.187,0.612


**Observation:** The final 23 features have substantially different scales, with ranges spanning from `11` (`month`) to over `7,500` (`co_today`). Scaling is therefore necessary for **Ridge Regression** and the **Dense NN**, while **Random Forest** can use the original feature values. Several pollutant and variability features are also strongly right-skewed, but this reflects the natural distribution of the collected air-quality data and does not by itself justify transforming or removing values. The appropriate scaler for the non-tree models still needs to be determined through further analysis.

In [4]:
print("NaN values:")
print(X_train.isna().sum()[X_train.isna().sum() > 0])

print("\nInfinite values:")
print(np.isinf(X_train.select_dtypes(include=np.number)).sum())

NaN values:
Series([], dtype: int64)

Infinite values:
pm25_today          0
pm10_today          0
o3_today            0
co_today            0
no2_today           0
so2_today           0
aqi_roll_mean_7     0
pm25_lag_1          0
pm25_lag_2          0
pm25_lag_3          0
pm25_roll_mean_3    0
pm25_roll_mean_7    0
pm25_roll_std_3     0
pm25_roll_std_7     0
pm10_roll_mean_7    0
pm10_roll_std_3     0
pm10_roll_std_7     0
o3_lag_1            0
o3_lag_2            0
o3_roll_mean_3      0
o3_roll_mean_7      0
o3_roll_std_3       0
month               0
dtype: int64


**Observation:** The training feature set contains **no missing values** and **no infinite values** across all 23 selected features. Therefore, no missing-value imputation or non-finite value handling is required before model training.

**Observation:** The final 23 features have substantially different scales, so **RobustScaler** was selected for Ridge Regression and the Dense NN because it is less sensitive to the strong skewness and extreme values present in several pollutant features. Random Forest will use the original feature values. Features with very high right-skewness will additionally use **log1p transformation** before scaling to reduce the influence of extreme magnitudes while preserving the underlying observations.